# The CUDA Programming Model

Companion notebook for the [CUDA Programming Model lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/02-cuda-programming-model).

**The idea in one sentence.** You write a **kernel** — a function that runs once per
**thread** — and launch a whole **grid** of thread-blocks over your data; each thread
computes its own global index from `blockIdx * blockDim + threadIdx` and works on one
element.

The three idioms every CUDA program uses, emulated in pure Python:

- **The global-index formula** — how a thread finds its data.
- **The bounds guard** (`if i < n`) — because you almost always launch *more* threads
  than elements, and the extras must do nothing.
- **The grid-stride loop** — let a fixed grid cover an array of *any* size.

We **validate that the index mapping covers the data exactly and the grid-stride loop
is correct for any launch size**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np

## 1 — A kernel is a per-thread function

In CUDA you write the body for *one* thread; the runtime launches thousands. We mimic that: a
`launch` driver loops over every (block, thread) pair and calls the kernel with that thread's
coordinates. The kernel computes its **global index** and writes one output element.

In [ ]:
def launch(kernel, grid_dim, block_dim, *args):
    """Emulate <<<grid_dim, block_dim>>> by calling `kernel` once per thread."""
    for block_idx in range(grid_dim):
        for thread_idx in range(block_dim):
            kernel(block_idx, block_dim, thread_idx, *args)

def vec_add(blockIdx, blockDim, threadIdx, A, B, C, n):
    i = blockIdx * blockDim + threadIdx        # the global-index formula
    if i < n:                                  # bounds guard for over-launch
        C[i] = A[i] + B[i]

n = 1000
A = np.arange(n, dtype=np.float32)
B = np.arange(n, dtype=np.float32) * 10
C = np.zeros(n, dtype=np.float32)

threads = 256
blocks = (n + threads - 1) // threads          # ceil(n / threads)
print(f"launching {blocks} blocks x {threads} threads = {blocks*threads} threads for n={n}")
launch(vec_add, blocks, threads, A, B, C, n)

assert np.allclose(C, A + B)
print("\u2713 vec_add kernel matches A + B")

### Validate: the launch covers every element exactly once

The point of the grid launch is that the active threads (those passing `i < n`) map
one-to-one onto the data indices $0..n-1$. We collect every global index written and
confirm it's exactly $\{0,\dots,n-1\}$ — no gaps (unwritten output) and no duplicates
(race conditions).

In [ ]:
written = []
def record_kernel(blockIdx, blockDim, threadIdx, n):
    i = blockIdx * blockDim + threadIdx
    if i < n:
        written.append(i)
launch(record_kernel, blocks, threads, n)
print(f'launched {blocks*threads} threads; {len(written)} passed the bounds guard for n={n}')
assert sorted(written) == list(range(n)), 'active threads must cover 0..n-1 exactly once'
assert len(written) == len(set(written)), 'no index written twice (no races)'
print('✅ the grid launch maps active threads one-to-one onto the data')

## 2 — Why the bounds guard matters

We launched `blocks*threads = 1024` threads for only `n = 1000` elements. The last 24 threads have
`i >= n`. Without `if (i < n)` they would write out of bounds. Let's see exactly which threads idle.

In [ ]:
active, idle = 0, 0
def count_kernel(blockIdx, blockDim, threadIdx, n):
    global active, idle
    i = blockIdx * blockDim + threadIdx
    if i < n:
        active += 1
    else:
        idle += 1

launch(count_kernel, blocks, threads, n)
print(f"launched threads: {blocks*threads}")
print(f"active (i < n):   {active}")
print(f"idle   (i >= n):  {idle}   <- these would corrupt memory without the guard")

## 3 — The grid-stride loop

Hard-coding one element per thread couples the launch size to `n`. The **grid-stride loop** lets each
thread handle multiple elements, striding by the total number of threads in the grid — so *any* grid
size is correct. We launch a deliberately small grid and still cover all `n` elements.

In [ ]:
def vec_add_stride(blockIdx, blockDim, threadIdx, gridDim, A, B, C, n):
    stride = blockDim * gridDim                # total threads in the grid
    i = blockIdx * blockDim + threadIdx
    while i < n:
        C[i] = A[i] + B[i]
        i += stride

C2 = np.zeros(n, dtype=np.float32)
small_blocks, small_threads = 4, 32            # only 128 threads for n=1000!
for b in range(small_blocks):
    for t in range(small_threads):
        vec_add_stride(b, small_threads, t, small_blocks, A, B, C2, n)

assert np.allclose(C2, A + B)
print(f"{small_blocks*small_threads} threads covered all {n} elements via grid-stride")
print(f"each thread handled ~{n // (small_blocks*small_threads)} elements")

### Validate: the grid-stride loop is correct for *any* launch size

A grid-stride loop lets a *small* grid process a *large* array by having each thread
handle multiple elements `stride = blockDim*gridDim` apart. We verify it produces the
right answer across wildly different launch configurations — the property that makes
kernels launch-size-independent.

In [ ]:
for gb, gt in [(1, 32), (4, 32), (8, 128), (100, 256)]:
    out = np.zeros(n, dtype=np.float32)
    for b in range(gb):
        for t in range(gt):
            vec_add_stride(b, gt, t, gb, A, B, out, n)
    ok = np.allclose(out, A + B)
    print(f'grid {gb:>3} x {gt:>3} = {gb*gt:>5} threads for n={n}: correct = {ok}')
    assert ok, 'grid-stride must give the right answer regardless of launch size'
print('\n✅ one kernel, any launch configuration — the grid-stride loop handles all sizes')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **missing bounds guard** | over-launched threads write out of range → corruption (demo) |
| **wrong index formula** | off-by-`blockDim` bugs leave gaps or double-write (races) |
| **block size not a multiple of 32** | partial warps waste lanes; pick multiples of the warp size |
| **assuming a fixed grid fits the data** | use a grid-stride loop instead of hard-coding the launch |
| **inter-block synchronization** | blocks can't sync mid-kernel; needs a new launch or cooperative groups |

Demo: without the bounds guard, the over-launched threads write past the array.

In [ ]:
# The bug the bounds guard prevents: without 'if i < n', over-launched threads write
# PAST the end of the array. We count how many out-of-range writes a guardless kernel
# would attempt — every one is undefined behaviour / memory corruption on a real GPU.
oob = 0
def guardless(blockIdx, blockDim, threadIdx, n):
    global oob
    i = blockIdx * blockDim + threadIdx
    if i >= n:              # (a real kernel WITHOUT the guard would write C[i] here)
        oob += 1
launch(guardless, blocks, threads, n)
print(f'launched {blocks*threads} threads for n={n}')
print(f'threads with i >= n that would corrupt memory without the guard: {oob}')
assert oob == blocks*threads - n
print('\nYou almost always launch ceil(n/block)*block > n threads, so the bounds guard is mandatory.')

## ✏️ Your turn

**Exercise.** Implement `global_indices(grid_dim, block_dim, n)` returning the list of global indices
`i = blockIdx*blockDim + threadIdx` for **only the active threads** (those with `i < n`), in launch
order. This is the set of elements a one-element-per-thread kernel actually processes.

In [ ]:
def global_indices(grid_dim, block_dim, n):
    out = []
    for blockIdx in range(grid_dim):
        for threadIdx in range(block_dim):
            # TODO(you): compute the global index and append it only if it is in bounds
            ...
    return out

In [ ]:
# This assert cell passes silently when your implementation is correct.
idx = global_indices(4, 256, 1000)
assert idx == list(range(1000)), "active threads should cover 0..999 exactly once"
assert global_indices(1, 8, 5) == [0, 1, 2, 3, 4]          # last 3 threads idle
assert len(global_indices(4, 256, 1000)) == 1000            # 24 of the 1024 threads idle
print("\u2713 global index mapping is correct, bounds guard drops the over-launched threads")

<details>
<summary>Solution</summary>

```python
def global_indices(grid_dim, block_dim, n):
    out = []
    for blockIdx in range(grid_dim):
        for threadIdx in range(block_dim):
            i = blockIdx * block_dim + threadIdx
            if i < n:
                out.append(i)
    return out
```

Every framework that dispatches to the GPU computes this same mapping under the hood. When you write
`a + b` in PyTorch, a pre-written CUDA kernel runs exactly this index math across the tensor.

</details>

## Key takeaways

- **A kernel runs once per thread;** each thread finds its element via
  `blockIdx*blockDim + threadIdx`.
- **Always launch a whole number of blocks**, so you over-launch — the **bounds
  guard** `if i < n` makes the extra threads no-ops (without it: memory corruption,
  demo).
- **The launch maps active threads one-to-one onto the data** — we verified full
  coverage, no gaps, no duplicate writes.
- **The grid-stride loop** decouples the kernel from the array size: a fixed grid
  correctly processes any `n` (verified across launch configs).